# Jupyter Notebook: Svalbard Environmental Data Downloader

**Objective:** This notebook provides the code to download historical weather, wave, and sea ice data for the Ny-Ålesund / Kongsfjorden area for February 2023. This data can then be correlated with the NTNU DAS dataset.

**Data Sources:**
1.  **MET Norway (Frost API):** For local weather station data (wind, temperature, etc.).
2.  **Copernicus Climate Data Store (CDS API):** For ERA5 reanalysis data, which includes ocean wave information.
3.  **Norwegian Polar Institute (NPI):** For sea ice charts (manual download).
4.  **Norwegian Water Resources and Energy Directorate (NVE):** For historical river discharge data.
5.  **Kartverket (Norwegian Mapping Authority):** For tidal data.
6.  **ERDDAP (Environmental Research Division's Data Access Program):** For oceanographic data.

## 1. Setup and Configuration

First, we'll install and import the necessary libraries and define our area of interest.

In [ ]:
%pip install requests pandas cdsapi lxml

import requests
import pandas as pd
import cdsapi
import os
import numpy as np
import xml.etree.ElementTree as ET
import io

DATA_FOLDER="weather_data/"
if not os.path.exists(DATA_FOLDER):
    os.makedirs(DATA_FOLDER)

### 1.1. Define Bounding Box
Define the geographical area for which to download data. We will use the coordinates from the NTNU DAS dataset to create a bounding box.

In [ ]:
# Create a dummy CSV for demonstration if the actual file is not present
if not os.path.exists('DAS NTNU/CGF_Svalbard_Approximate_LatLon_coordinates.csv'):
    if not os.path.exists('DAS NTNU'):
        os.makedirs('DAS NTNU')
    dummy_coords = pd.DataFrame({
        'lat': ['lat', 78.9423377, 78.93784614412505],
        'lon': ['lon', 11.8860757, 11.909645816599996]
    })
    dummy_coords.to_csv('DAS NTNU/CGF_Svalbard_Approximate_LatLon_coordinates.csv', index=False)

coords_df = pd.read_csv('DAS NTNU/CGF_Svalbard_Approximate_LatLon_coordinates.csv')
# Remove the first row if it contains column headers as data
coords_df = coords_df.iloc[1:].reset_index(drop=True)

# Convert lat/lon columns to float for calculations
coords_df['lat'] = coords_df['lat'].astype(float)
coords_df['lon'] = coords_df['lon'].astype(float)

# Set the bounding box size in degrees for ~500 meters
bbox_size_lat = 500 / 111320
mean_lat = coords_df['lat'].mean()
meters_per_deg_lon = 111320 * np.cos(np.deg2rad(mean_lat))
bbox_size_lon = 500 / meters_per_deg_lon

# Define a bounding box around the cable for area-based data requests
bbox = {
    'north': coords_df['lat'].iloc[0] + bbox_size_lat,
    'south': coords_df['lat'].iloc[0] - bbox_size_lat,
    'east': coords_df['lon'].iloc[0] + bbox_size_lon,
    'west': coords_df['lon'].iloc[0] - bbox_size_lon
}
print(f"Cable bounding box defined: {bbox}")

### 1.2. API Credentials

**IMPORTANT:** You must replace the placeholder values below with your own credentials.

1.  **Frost API (MET Norway):** Get your Client ID and Client Secret from [https://frost.met.no/auth/requestCredentials.html](https://frost.met.no/auth/requestCredentials.html).
2.  **CDS API (Copernicus):** Get your API key from [https://cds.climate.copernicus.eu/api-how-to](https://cds.climate.copernicus.eu/api-how-to) and place it in a `.cdsapirc` file in your home directory.
3. **NVE API (Norwegian Water Resources and Energy Directorate):** Request an API key at [https://hydapi.nve.no/](https://hydapi.nve.no/).

In [ ]:
FROST_CLIENT_ID = 'YOUR_CLIENT_ID_HERE'
FROST_CLIENT_SECRET = 'YOUR_SECRET_KEY_HERE'
NVE_API_KEY = 'YOUR_NVE_API_KEY_HERE'

## 2. MET Norway - Frost API (Local Weather)

Fetch hourly weather data from the Ny-Ålesund station (`SN99910`), the closest official weather station to the DAS cable.

In [ ]:
if FROST_CLIENT_ID == 'YOUR_CLIENT_ID_HERE':
    print("Skipping MET Norway download. Please add your Frost API Client ID and Secret.")
else:
    endpoint = 'https://frost.met.no/observations/v0.jsonld'
    station_id = 'SN99910' # Ny-Ålesund II
    start_date = '2023-02-01'
    end_date = '2023-03-01'
    elements = 'wind_speed,wind_from_direction,air_temperature,air_pressure_at_sea_level'

    params = {
        'sources': station_id,
        'referencetime': f'{start_date}/{end_date}',
        'elements': elements,
    }

    print("Downloading data from MET Norway Frost API...")
    r = requests.get(endpoint, params, auth=(FROST_CLIENT_ID, FROST_CLIENT_SECRET))
    
    if r.status_code == 200:
        json_data = r.json()
        df = pd.json_normalize(json_data['data'], record_path=['observations'], meta=['referenceTime', 'sourceId'])
        df = df.rename(columns={'referenceTime': 'time', 'value': 'value', 'elementId': 'element'})
        
        df_pivot = df.pivot_table(index='time', columns='element', values='value').reset_index()
        
        output_filename = os.path.join(DATA_FOLDER, 'met_norway_nyalesund_2023-02.csv')
        df_pivot.to_csv(output_filename, index=False)
        print(f"Successfully saved weather data to {output_filename}")
    else:
        print(f"Error fetching data from Frost API: {r.status_code}")
        print(r.json())

## 3. Copernicus CDS API (ERA5 Wave Data)

Download hourly ocean wave data from the ERA5 global reanalysis dataset. This provides context on the broader ocean state and swells.

In [ ]:
if not os.path.exists(os.path.expanduser('~/.cdsapirc')):
    print("Skipping Copernicus download. `.cdsapirc` file not found.")
    print("Please create it with your CDS UID and API Key. See: https://cds.climate.copernicus.eu/api-how-to")
else:
    print("Downloading ERA5 wave data from Copernicus CDS API...")
    print("This may take a few minutes.")

    c = cdsapi.Client()

    output_filename_era5 = os.path.join(DATA_FOLDER, 'era5_waves_svalbard_2023-02.nc')

    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'format': 'netcdf',
            'variable': [
                'mean_wave_direction', 'mean_wave_period', 'significant_height_of_combined_wind_waves_and_swell',
            ],
            'year': '2023',
            'month': '02',
            'day': [f'{i:02d}' for i in range(1, 29)],
            'time': [f'{h:02d}:00' for h in range(24)],
            'area': [
                bbox['north'], bbox['west'], bbox['south'], bbox['east'],
            ],
        },
        output_filename_era5
    )
    print(f"Successfully saved ERA5 wave data to {output_filename_era5}")

## 4. Norwegian Polar Institute (Sea Ice Data)

NPI provides sea ice charts for Kongsfjorden. This data is typically downloaded manually.

**Action Required:**
1.  Go to the [NPI Kongsfjorden Sea Ice Dataset](https://data.npolar.no/dataset/d6d31f5b-8413-42b4-9736-db88d55816dc).
2.  Download the data for the relevant period (e.g., as a CSV).
3.  Place the downloaded file in the `weather_data/` directory.

The code below assumes you have downloaded a CSV file named `npi_sea_ice_kongsfjorden.csv`.

In [ ]:
npi_filename = os.path.join(DATA_FOLDER, 'npi_sea_ice_kongsfjorden.csv')
try:
    df_ice = pd.read_csv(npi_filename)
    print("Successfully loaded NPI sea ice data:")
    print(df_ice.head())

except FileNotFoundError:
    print(f"Sea ice file '{npi_filename}' not found.")
    print("Please download it from the NPI data portal and place it in the weather_data/ directory.")

# 5. Norwegian Water Resources and Energy Directorate (NVE) - River Discharge
Historical river discharge data for the Bayelva river, which flows into Kongsfjorden. This data can provide insights into freshwater runoff effects.

In [ ]:
if NVE_API_KEY == 'YOUR_NVE_API_KEY_HERE':
    print("Skipping NVE download. Please add your NVE API Key.")
else:
    nve_endpoint = 'https://hydapi.nve.no/api/v1/Observations'
    bayelva_station_id = '400.1.0' # Station ID for Bayelva
    discharge_parameter = '1001' # Parameter for Mean Discharge (m3/s)
    start_date_nve = '2023-02-01'
    end_date_nve = '2023-02-28'

    headers = {'X-API-Key': NVE_API_KEY}
    params = {
        'StationId': bayelva_station_id,
        'Parameter': discharge_parameter,
        'ResolutionTime': '60',
        'ReferenceTime': f'{start_date_nve}/{end_date_nve}'
    }

    print(f"Downloading river discharge data for station {bayelva_station_id}...")
    try:
        response = requests.get(nve_endpoint, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        if data.get('data'):
            observations = data['data'][0].get('observations', [])
            if observations:
                df_nve = pd.DataFrame(observations)
                output_filename_nve = os.path.join(DATA_FOLDER, f"nve_data_{bayelva_station_id}_discharge.csv")
                df_nve.to_csv(output_filename_nve, index=False)
                print(f"Successfully saved NVE data to {output_filename_nve}")
            else:
                print("No observations found for the specified period.")
        else:
            print("No data returned from NVE API.")
    except requests.exceptions.RequestException as e:
        print(f"Error fetching NVE data: {e}")

# 6. Kartverket - Tidal Data
Download tidal data for the Ny-Ålesund area.

In [ ]:
print("Downloading tidal data from Kartverket API...")
tides_endpoint = 'https://vannstand.kartverket.no/tideapi.php'
tides_params = {
    'lat': str(coords_df['lat'].iloc[0]),
    'lon': str(coords_df['lon'].iloc[0]),
    'fromtime': '2023-02-01T00:00',
    'totime': '2023-03-01T00:00',
    'datatype': 'all',
    'refcode': 'cd',
    'lang': 'en',
    'interval': '10',
    'tide_request': 'locationdata'
}

try:
    response = requests.get(tides_endpoint, params=tides_params)
    response.raise_for_status()
    root = ET.fromstring(response.content)
    
    waterlevel_data = []
    for wl in root.findall('.//waterlevel'):
        waterlevel_data.append(wl.attrib)
        
    df_waterlevel = pd.DataFrame(waterlevel_data)

    if not df_waterlevel.empty:
        output_filename_tides = os.path.join(DATA_FOLDER, 'kartverket_tides_nyalesund_2023-02.csv')
        df_waterlevel.to_csv(output_filename_tides, index=False)
        print(f"Successfully saved tidal data to {output_filename_tides}")
    else:
        print("No tidal data returned.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching data from Kartverket API: {e}")
except ET.ParseError as e:
    print(f"Error parsing XML: {e}")

# 7. ERDDAP - Oceanographic Data
Download oceanographic data from ERDDAP servers. The following example downloads data from the IADC ERDDAP server.

In [ ]:
erddap_datasets = {
    "iadc_707mqt_feb2023": "https://data.iadc.cnr.it/erddap/tabledap/707mqt.csv?time,latitude,longitude,depth,TEMP,TEMP_QC,CurrVelE_ADCP,CurrVelN_ADCP,CurrVelUp_ADCP,Curr_QC,station_id&time>=2023-02-01T00:00:00Z&time<=2023-03-01T00:00:00Z",
    "iadc_aa8850ba_feb2023": "https://data.iadc.cnr.it/erddap/tabledap/aa8850ba-e02c-4f11-b225-22f3c4aa07cb.csv?time,latitude,longitude,depth,sea_water_pressure,sea_water_pressure_QC,TEMP,TEMP_QC,COND,COND_QC,PSAL,PSAL_QC,SIGT,SIGT_QC,DOXY,DOXY_QC,TSED,TSED_QC,station_id&time>=2023-02-01T00:00:00Z&time<=2023-03-01T13:00:02Z"
}

for name, url in erddap_datasets.items():
    print(f"Downloading {name} from ERDDAP...")
    try:
        df_erddap = pd.read_csv(url, skiprows=[1]) # Skip the second line which often contains units
        print(f"Downloaded {len(df_erddap)} rows.")
        output_filename_erddap = os.path.join(DATA_FOLDER, f"{name}.csv")
        df_erddap.to_csv(output_filename_erddap, index=False)
        print(f"Saved ERDDAP data to {output_filename_erddap}")
    except Exception as e:
        print(f"Failed to download {name}: {e}")